# F1 Qualifying Monte Carlo Simulation

## Setup

### Packages

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import fastf1 as f1
import matplotlib.pyplot as plt
import seaborn as sns

### Setting up FastF1 cache (optional)

In [ ]:
import os

# Create cache directory if it doesn't exist
file_path = 'your file path goes here'
cache_dir = os.path.expanduser(file_path)
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

# Enable the cache
f1.Cache.enable_cache(cache_dir)

# Check cache is correct
print(f1.Cache)

## Functions

### Monte Carlo simulation functions

In [ ]:
def create_driver_stats(session):
    ''''
    Takes session as input.
    Filters to only competitive, legal laps (within 107% of fastest).
    Creates LapTimeSeconds column.
    Exports driver lap time mean, standard deviation and count.
    Drivers only with 1 lap in session have median std.
    '''

    laps = session.pick_quicklaps().loc[lambda df: ~df['Deleted']].copy()
    laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

    driver_stats = laps.groupby('Driver')['LapTimeSeconds'].aggregate(['mean', 'std', 'count'])

    backup_std = driver_stats['std'].median()
    driver_stats['std'] = driver_stats['std'].fillna(backup_std)

    return driver_stats

In [ ]:
def simulate_qualifying(driver_stats, n_attempts=2):
    '''
    Takes driver stats contain name, mean and std as input.
    Default number of attempts is 2, as is typical in Q3.
    Uses normal distribution with mean and std to generate a lap time.
    Returns list of drivers sorted by position, P1 first.
    '''

    results = {}

    for driver in driver_stats.index:
        mean = driver_stats.loc[driver, 'mean']
        std = driver_stats.loc[driver, 'std']

        lap_times = np.random.normal(mean, std, n_attempts)
        best_lap = lap_times.min()

        results[driver] = best_lap

    sorted_drivers = sorted(results, key=results.get)

    return sorted_drivers

In [ ]:
def qualifying_MC(driver_stats, n=500, n_attempts=2):
    '''
    Takes driver stats contain name, mean and std as input, and number of simulations, default 500.
    Returns dictionary containing drivers as keys, and values as dictionaries with counts on positions.
    '''

    from collections import defaultdict

    # Automically create position with count 0 if not seen before, with fresh dictionary for each driver.
    position_counts = defaultdict(lambda: defaultdict(int))

    for i in range(n):
        result = simulate_qualifying(driver_stats, n_attempts=n_attempts)

        # Add count for that position for driver, adjust for 0 indexing.
        for position, driver in enumerate(result):
            position_counts[driver][position + 1] +=1

    return position_counts

In [ ]:
def get_position_probability(position_counts, n=500):
    '''
    Takes position counts in and returns counts as probabilities.
    '''
    position_probabilities = {}

    for driver, positions in position_counts.items():
        position_probabilities[driver] = {
            position: count / n
            for position, count in positions.items()
        }

    return position_probabilities

In [ ]:
def monte_carlo_qualifying(gp, year, n=500):
    '''
    Input is GP name, year, and number of simulations n.
    Returns dataframe of drivers with probabilities for each position.
    '''
    if n < 1:
        raise ValueError('n must be a positive integer.')
    
    session = f1.get_session(year, gp, 'Q')
    session.load()

    q1, q2, q3 = session.laps.split_qualifying_sessions()

    driver_stats = create_driver_stats(q3)

    position_counts = qualifying_MC(driver_stats, n)
    position_probabilities = get_position_probability(position_counts, n)

    df = pd.DataFrame.from_dict(position_probabilities, orient='index')

    df = df.fillna(0)
    df.index.name = 'Driver'
    df.columns.name = 'Position'
    df = df.reindex(sorted(df.columns), axis=1)

    return df

### Visualisation functions

In [ ]:
def get_driver_colours(session):
    '''
    Input is session. Returns dictionary of drivers in session with corresponding team colours.
    '''

    laps = session.laps

    # Create dictionary of drivers with their teams.
    driver_teams = laps[['Driver', 'Team']].drop_duplicates().set_index('Driver')['Team'].to_dict()

    # Map teams to colours, so each driver has their team colour in dictionary.
    driver_colours = {}
    for driver in driver_teams.keys():
        driver_colours[driver] = f1.plotting.get_team_color(driver_teams[driver], session=session)

    return driver_colours

In [ ]:
def f1_plot_theme():
    '''
    Sets style for matplotlib plots.
    '''
    plt.style.use("dark_background")
    plt.rcParams["figure.facecolor"] = "#1b1b1b"
    plt.rcParams["axes.facecolor"] = "#1b1b1b"
    plt.rcParams["grid.color"] = "#898989"
    plt.rcParams["axes.grid.axis"] = "y"
    plt.rcParams["text.color"] = "#898989"
    plt.rcParams["axes.labelcolor"] = "#d6d6d6"
    plt.rcParams["axes.titlecolor"] = "#f2f2f2"
    plt.rcParams["xtick.color"] = "#a4a4a4"
    plt.rcParams["ytick.color"] = "#a4a4a4"
    plt.tight_layout(pad=2)

In [ ]:
def position_probability_plot(gp, year, pos=1, n=500):
    '''
    Takes in session data, qualifying position (default is pole) and number of simulations.
    Returns probability distribution plot for given position.
    '''
    
    import fastf1.plotting
    fastf1.set_log_level('ERROR')

    print(f"Running {n} qualifying simulations...")

    session = f1.get_session(year, gp, 'Q')
    session.load()

    driver_colours = get_driver_colours(session)

    df = monte_carlo_qualifying(gp, year, n)

    data = df[pos].reset_index()
    data.columns = ["Driver", "Probability"]
    data = data.sort_values("Probability", ascending=False)
    data['Colour'] = data['Driver'].map(driver_colours)


    plt.figure(figsize=(12, 6))
    
    plt.bar(
        data['Driver'],
        data['Probability'],
        color=data['Colour']
    )

    f1_plot_theme()
    sns.despine()

    plt.title(f"{gp} {year} P{pos} Qualifying Probability Distribution - {n} simulations", fontsize=16, weight="bold", pad=15)
    plt.xlabel('Driver', fontsize=12, weight="bold", labelpad=15)
    plt.ylabel('Probability', fontsize=12, weight="bold", labelpad=15)

    return plt.show()

In [4]:
def expected_position(gp, year, driver, n=500):
    '''
    Takes gp info, driver abbreviation and number of simulations.
    Returns driver position probability distribution.
    '''
    

    print(f"Running {n} qualifying simulations...")

    session = f1.get_session(year, gp, 'Q')
    session.load()

    driver_colours = get_driver_colours(session)

    df = monte_carlo_qualifying(gp, year, n)

    row = df.loc[driver]
    positions = sorted(row.index.astype(int))

    probabilities = row.values

    plt.figure(figsize=(12, 6))

    plt.bar(
        positions,
        probabilities,
        color = driver_colours[driver]
    )

    f1_plot_theme()
    sns.despine()

    plt.title(f"{gp} {year} {driver} Qualifying Position Probability Distribution - {n} simulations", fontsize=16, weight="bold", pad=15)
    plt.xlabel('Position', fontsize=12, weight="bold", labelpad=15)
    plt.ylabel('Probability', fontsize=12, weight="bold", labelpad=15)
    plt.xticks(positions)

    return plt.show()